# Cat Health Agentic RAG with LangChain and LangGraph

In Session 1, retrieval happened before every answer:

```text
question -> retrieve -> generate
```

In this notebook, retrieval becomes a **tool**. The agent can call that tool when it decides the user's question needs cat health guideline context.

That is the core idea of agentic RAG for this session:

```text
question -> agent decides whether to retrieve -> optional retrieval tool call -> answer
```

We will show that loop two ways:

1. **High-level LangChain path**: use `create_agent` with middleware.
2. **Explicit LangGraph path**: build the same loop with `StateGraph`, `ToolNode`, and `tools_condition`.

Both versions use the same retriever tool. The point is to see that agentic RAG is about giving the agent retrieval as an action, not forcing retrieval as a pre-step.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how agentic RAG differs from a fixed two-step RAG pipeline.
- Build and inspect a retrieval tool over a Qdrant vector store.
- Use LangChain middleware to observe and constrain an agent loop.
- Compare the convenience of `create_agent` with the control of an explicit LangGraph graph.
- Design focused routing and middleware experiments for an agentic RAG system.

## Table of Contents

- **Breakout Room #1: High-Level Agentic RAG with LangChain**
  - Task 1: Environment Setup
  - Task 2: Load and Index the Cat Health Corpus
  - Task 3: Create a Retriever Tool
  - Task 4: Build an Agent with `create_agent` and Middleware
  - Task 5: Visualize and Stream the `create_agent` Agent
  - 🏗️ Activity #1: Add a Retriever Tool-Call Budget
- **Breakout Room #2: Explicit Agent Loop with LangGraph**
  - Task 6: Build the Same Agent Loop with LangGraph
  - 🏗️ Activity #2: Add Deterministic Scope Routing
  - 🚧 Advanced Build: Add Explicit Retrieval Quality Control

---
# Breakout Room #1
## High-Level Agentic RAG with LangChain

In this breakout room, you will build the shared retrieval tool, give it to `create_agent`, and use middleware and streaming to inspect and constrain the agent loop.

## Task 1: Environment Setup

From the `02_Agentic_RAG_LangGraph_LangChain` folder, install dependencies with uv:

```bash
uv sync
```

Then open this notebook in Cursor or VS Code and select the Python/Jupyter environment created by uv.

### Imports

LangChain gives us document loading, splitting, embeddings, Qdrant integration, tools, models, and the high-level agent loop.

In [ ]:
from pathlib import Path
from getpass import getpass
import os

from IPython.display import Image, display

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware, ToolCallLimitMiddleware, before_model
from langchain.tools import tool
from langchain_community.document_loaders import TextLoader
from langchain_core.messages import SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition

### API Keys and Models

The chat model and embedding model both use OpenAI. If `OPENAI_API_KEY` is not already set in your environment, this cell will ask for it securely.

LangSmith tracing is optional. If you set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY`, LangChain/LangGraph calls will be traced automatically.

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

os.environ.setdefault("LANGSMITH_PROJECT", "aim-session-3-agentic-rag")

chat_model_name = os.environ.get("AIM_CHAT_MODEL", "gpt-5.4-mini")
embedding_model_name = os.environ.get("AIM_EMBEDDING_MODEL", "text-embedding-3-small")

llm = ChatOpenAI(model=chat_model_name)
embeddings = OpenAIEmbeddings(model=embedding_model_name)

print(f"Chat model: {chat_model_name}")
print(f"Embedding model: {embedding_model_name}")
print(f"LangSmith tracing: {os.environ.get('LANGSMITH_TRACING', 'false')}")

## Task 2: Load and Index the Cat Health Corpus

We will use a small course-owned Markdown corpus instead of a PDF. This keeps the session focused on the agentic RAG pattern instead of PDF parsing.

**Further Reading:**
- [LangChain Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)
- [Qdrant LangChain Integration](https://qdrant.tech/documentation/frameworks/langchain/)

In [ ]:
corpus_path = Path("data/cat_health_guidelines.md")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the cat health corpus at: {corpus_path.resolve()}\n"
        "Run this notebook from the 02_Agentic_RAG_LangGraph_LangChain folder."
    )

loader = TextLoader(str(corpus_path), encoding="utf-8")
documents = loader.load()

for document in documents:
    document.metadata["source"] = corpus_path.name
    document.metadata["document_type"] = "cat_health_guidelines"

print(f"Loaded {len(documents)} document(s).")
print(documents[0].page_content[:800])
print("\nMetadata:", documents[0].metadata)

### Split the Corpus

Chunks should be large enough to keep a useful idea together, but small enough that retrieval returns focused context.

In [ ]:
chunk_size = 900
chunk_overlap = 120

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap,
    add_start_index=True,
    separators=["\n## ", "\n### ", "\n\n", "\n", ". ", " "],
)

splits = text_splitter.split_documents(documents)

for index, split in enumerate(splits):
    split.metadata["chunk_id"] = index

print(f"Created {len(splits)} chunks.")
print(splits[0].page_content[:800])
print("\nMetadata:", splits[0].metadata)

### Build the Qdrant Vector Store

For the course notebook, Qdrant runs in memory. There is no Docker service or cloud account required, and the collection disappears when the notebook kernel stops.

In [ ]:
collection_name = "cat_health_agentic_rag"

vector_store = QdrantVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    location=":memory:",
    collection_name=collection_name,
)

retrieval_k = 4
print(f"Built in-memory Qdrant collection: {collection_name}")

## Task 3: Create a Retriever Tool

This is the important shift from Session 1.

The retriever is no longer a required pre-step. It is now a tool the agent can call when it wants context from the cat health guideline corpus.

The tool name, docstring, inputs, and output format form a contract with the model. Clear contracts make good tool-selection and grounded-answer behavior more likely.

**Further Reading:**
- [LangChain Tools](https://docs.langchain.com/oss/python/langchain/tools)
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629)

In [ ]:
def _format_retrieved_docs(scored_docs: list[tuple]) -> str:
    formatted_chunks = []
    for index, (doc, score) in enumerate(scored_docs, start=1):
        source = doc.metadata.get("source", "unknown")
        chunk_id = doc.metadata.get("chunk_id", "unknown")
        start_index = doc.metadata.get("start_index", "unknown")
        score_text = f"{score:.3f}" if isinstance(score, (float, int)) else str(score)
        formatted_chunks.append(
            f"[Source {index}: {source}, chunk_id={chunk_id}, start_index={start_index}, score={score_text}]\n"
            f"{doc.page_content.strip()}"
        )
    return "\n\n".join(formatted_chunks)


@tool
def retrieve_cat_health_guidelines(query: str) -> str:
    """Search the cat health guideline corpus for relevant context about cat preventive care, nutrition, hydration, vaccines, parasites, dental health, urinary warning signs, emergencies, senior cats, stress, behavior, and safe home monitoring."""
    results = vector_store.similarity_search_with_score(query, k=retrieval_k)
    if not results:
        return "No relevant cat health guideline context found."
    return _format_retrieved_docs(results)


retriever_tool = retrieve_cat_health_guidelines

Try the tool directly once. This is just to understand what the agent will see when it calls the tool.

In [ ]:
print(
    retriever_tool.invoke(
        {"query": "What urinary signs suggest a cat needs urgent veterinary care?"}
    )[:2500]
)

#### ❓ Question #1

What changes when retrieval becomes a tool instead of a mandatory first step?

##### Answer:

_Retrieval moves from a fixed pipeline stage to a **decision the agent makes**. Instead of every turn running `question → retrieve → generate`, the model now decides **whether** to retrieve, **when**, and **with what query** — and it can call the tool zero times (greetings, out-of-scope, or follow-ups it can answer from the existing conversation) or several times with refined queries. The upside is efficiency and flexibility: no wasted retrieval/embedding cost on questions that don't need the corpus, and the ability to reformulate or do multiple targeted searches. The cost is that retrieval is now **non-deterministic and prompt-dependent** — the agent can fail to retrieve when it should (and answer ungrounded), or retrieve when it shouldn't, and its choices hinge on the tool's name/docstring and the system prompt. That makes grounding harder to guarantee and means you have to inspect the **agent's path** (did the tool actually fire?), not just judge the final answer._

## Task 4: Way 1 - Build an Agent with `create_agent` and Middleware

`create_agent` builds the agent loop for us:

1. The model reads the user question and available tools.
2. The model either answers directly or asks to call a tool.
3. If it asks for a tool, LangChain executes the tool.
4. The tool result is added back to the message history.
5. The model continues until it produces a final answer.

This is the fastest way to build agentic RAG: give the agent a retriever tool and let the agent decide when to call it.

Middleware hooks into the loop without requiring us to rebuild the graph. Below, custom `before_model` middleware logs each model step, while built-in middleware limits the number of model calls in one run.

**Further Reading:**
- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Middleware](https://docs.langchain.com/oss/python/langchain/middleware)

In [ ]:
AGENT_SYSTEM_PROMPT = """You are a cat health guideline assistant in an agentic RAG lesson.

You have one retrieval tool: retrieve_cat_health_guidelines.

Use the retrieval tool when the user asks about cat health, cat symptoms, preventive care, nutrition, vaccines, parasites, dental health, urinary signs, senior cats, stress, behavior, or home monitoring.

When you use retrieved context:
- Answer only from that retrieved context.
- Include a short Sources line using the source labels returned by the tool.
- Remind the user to contact a veterinarian for medical decisions, urgent symptoms, or worsening symptoms.

If the user asks something unrelated to cat health, do not call the tool. Briefly say this notebook is scoped to the cat health guideline corpus.
If the retrieved context does not contain enough information, say you do not have enough information in the cat health guidelines to answer.
"""


@before_model
def log_before_model(state, runtime):
    """Log a compact view of each model step in the agent loop."""
    print(f"[middleware] Calling the model with {len(state['messages'])} message(s).")


agent_middleware = [
    log_before_model,
    ModelCallLimitMiddleware(run_limit=4, exit_behavior="end"),
]


agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=AGENT_SYSTEM_PROMPT,
    middleware=agent_middleware,
)

print(type(agent))

#### ❓ Question #2

What does middleware let us change or observe without rebuilding the agent loop? Why is a model-call limit useful?

##### Answer:

_Middleware hooks into the **already-compiled** loop at defined points (`before_model`, `after_model`, around model/tool calls) so we can observe and constrain behavior without editing the graph. That covers cross-cutting concerns: logging/tracing each step (like `log_before_model` printing the message count), enforcing limits on model calls or tool calls, injecting or trimming messages, adding guardrails, and retries. It's behavior layered **around** the loop, reusable across agents, with the graph topology untouched._

_A model-call limit is useful as a **safety and cost guardrail**. An agent loop cycles model → tool → model → …, and a confused or adversarial run can keep calling tools and re-invoking the model indefinitely, burning tokens, money, and latency. `ModelCallLimitMiddleware(run_limit=4, exit_behavior="end")` caps the number of model invocations per run so the loop **terminates deterministically** instead of looping forever — here it ends gracefully at the cap rather than erroring. It bounds worst-case cost/latency and protects against runaway loops._

## Task 5: Visualize and Stream the `create_agent` Agent

`create_agent` returns a compiled LangGraph graph. We can visualize that graph and stream updates to inspect when the retriever tool was called.

The exact generated graph includes middleware nodes, but its core loop is:

```text
START -> before-model middleware -> model -> after-model middleware -> END
                                                |
                                                | tool call
                                                v
                                              tools
                                                |
                                                +----> loop back to before-model middleware
```

**Further Reading:**
- [LangGraph Streaming](https://docs.langchain.com/oss/python/langgraph/streaming)
- [LangSmith Observability](https://docs.langchain.com/langsmith/observability)

### Visualize the `create_agent` Graph

Run the next cell to render the exact compiled graph, including middleware nodes. If Mermaid PNG rendering is unavailable in your environment, the fallback prints the Mermaid source.

In [ ]:
try:
    display(Image(agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Could not render Mermaid PNG. Mermaid source follows:\n")
    print(agent.get_graph().draw_mermaid())
    print(f"\nRender error: {exc}")

### Stream Agent Runs

Streaming updates lets us inspect the path the agent took. Look for tool messages to see when retrieval happened.

In [ ]:
def print_agent_stream(question: str):
    """Run the agent and print each graph update."""
    inputs = {"messages": [{"role": "user", "content": question}]}

    for chunk in agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            if update is None:
                print("No state update returned.")
                continue

            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue

            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)

### Example 1: Cat Health Question

This should call the retrieval tool before answering.

In [ ]:
print_agent_stream("What urinary signs suggest my cat needs urgent veterinary care?")

### Example 2: Another Cat Health Question

This should also retrieve, but with a different search query.

In [ ]:
print_agent_stream("What preventive care should an adult cat get each year?")

### Example 3: Unrelated Question

This should not call the retrieval tool.

In [ ]:
print_agent_stream("Who won the 2022 FIFA World Cup?")

#### ❓ Question #3

For each example, did the agent call the retrieval tool? Why or why not?

##### Answer:

_The streamed updates show a `tools` node (with a `ToolMessage` from `retrieve_cat_health_guidelines`) for the first two examples and **no** `tools` node for the third:_

- _**Example 1 — "What urinary signs suggest my cat needs urgent veterinary care?"** → **Yes, retrieved.** This is squarely a cat-health/symptom question, exactly what the system prompt and the tool's docstring tell the model to look up. The model emits a tool call, the `tools` node runs, and the answer is built from the returned source-labeled context (with a vet reminder)._
- _**Example 2 — "What preventive care should an adult cat get each year?"** → **Yes, retrieved.** Also in-scope (preventive care), so the model issues a tool call — typically with a different search query than Example 1 — and answers from the retrieved chunks._
- _**Example 3 — "Who won the 2022 FIFA World Cup?"** → **No retrieval.** This is unrelated to cat health, and the prompt instructs the agent not to call the tool for off-topic questions. The model answers directly in the `agent` node (briefly noting the notebook is scoped to the cat health corpus), and no `tools` node appears in the stream._

_So retrieval tracked topicality: the agent called the tool only when the question fell inside the corpus's domain, which is the behavior the tool contract + system prompt were designed to produce. (Because the decision is model-driven, the exact query wording can vary run to run, but the retrieve/no-retrieve pattern is stable.)_

## 🏗️ Activity #1: Add a Retriever Tool-Call Budget

Middleware can enforce an operational rule without changing the retriever tool or rebuilding the agent graph. Create a second agent that allows at most **one** call to `retrieve_cat_health_guidelines` per run.

### Requirements

1. Create a `ToolCallLimitMiddleware` instance scoped to `retriever_tool.name` with `run_limit=1` and `exit_behavior="continue"`.
2. Create `budgeted_agent` with the same model, tool, prompt, and existing middleware plus the new retrieval budget.
3. Ask the agent to use separate searches for urinary emergency signs and annual preventive care before summarizing both.
4. Inspect the stream and explain what the middleware allowed or blocked.

**Further Reading:**
- [Built-in Middleware](https://docs.langchain.com/oss/python/langchain/middleware/built-in)

In [ ]:
# Activity #1 workspace
# Goal: allow at most ONE call to retrieve_cat_health_guidelines per run, using
# middleware only (no change to the tool or the graph).

# 1. Tool-call budget scoped to the retriever tool.
#    exit_behavior="continue" lets the agent keep going (and produce a final
#    answer) after the limit is hit, instead of ending or erroring.
retrieval_budget = ToolCallLimitMiddleware(
    tool_name=retriever_tool.name,
    run_limit=1,
    exit_behavior="continue",
)

# 2. Same model/tool/prompt/middleware as before, plus the new retrieval budget.
budgeted_agent = create_agent(
    model=llm,
    tools=[retriever_tool],
    system_prompt=AGENT_SYSTEM_PROMPT,
    middleware=agent_middleware + [retrieval_budget],
)


def print_budgeted_stream(question: str):
    """Stream the budgeted agent so we can see which tool calls run vs. get blocked."""
    inputs = {"messages": [{"role": "user", "content": question}]}

    for chunk in budgeted_agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            if update is None:
                print("No state update returned.")
                continue

            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue

            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)


# 3. Force the agent to want TWO separate searches in one run.
print_budgeted_stream(
    "Use one search to find urinary emergency warning signs, then a separate "
    "search to find recommended annual preventive care, and summarize both."
)

### 📝 Activity #1 Notes

- Which retrieval calls did the agent attempt?
- Which call did the middleware allow or block?
- What quality or safety trade-off does this budget introduce?

##### Answer:

_**Attempted:** Prompted to research two topics, the agent tries **two** separate `retrieve_cat_health_guidelines` calls — first for urinary emergency warning signs, then for annual preventive care._

_**Allowed vs. blocked:** With `run_limit=1`, the middleware **allows the first** retrieval call (you see one `tools` node run with a real `ToolMessage`) and **blocks the second**. Because `exit_behavior="continue"`, the loop doesn't end or raise — instead the second tool call is short-circuited (the agent gets a limit/blocked signal back in place of fresh context) and the model **continues to a final answer** using only the first search's results. So the stream shows: first search → real results, second search attempt → blocked, then a final summary._

_**Trade-off:** A hard one-call budget bounds cost and latency and prevents over-retrieval, but it can **starve the agent of needed evidence**. Here the question genuinely needs two distinct searches, so the second topic (preventive care) is answered from weaker, missing, or model-prior context rather than retrieved guideline text — raising the risk of an incomplete or less-grounded answer. That's the core safety/quality tension: tight budgets protect resources but can degrade grounding for multi-part questions. A middle ground (e.g. `run_limit=2`, or routing multi-topic questions differently) trades a little extra cost for better coverage._

## Breakout Room #1 Summary

In BOR1, you:

- Turned retrieval into a source-labeled tool the model can choose to call.
- Built a high-level agent loop with `create_agent`.
- Used middleware to observe the loop and constrain model or tool calls.
- Used streaming to inspect retrieval decisions instead of judging only the final answer.

---
# Breakout Room #2
## Explicit Agent Loop with LangGraph

In this breakout room, you will rebuild the same model-tools loop explicitly, then add routing behavior that would require graph-level control.

## Task 6: Way 2 - Build the Same Agent Loop with LangGraph

Now we will build the minimal agent loop ourselves.

This is the same idea as `create_agent`, but expressed directly as a graph:

```text
START -> agent model -------------------------------> END
              |
              | tool call
              v
            tools
              |
              +---------------------> agent model
```

There is still no mandatory pre-retrieval step. Retrieval only happens if the model emits a tool call. Unlike middleware, graph nodes and conditional edges let us change the control flow itself.

**Further Reading:**
- [LangGraph Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api)
- [LangGraph Workflows and Agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)

In [ ]:
llm_with_tools = llm.bind_tools([retriever_tool])


def call_model(state: MessagesState):
    """Call the model with tools bound so it can choose whether to retrieve."""
    response = llm_with_tools.invoke(
        [SystemMessage(content=AGENT_SYSTEM_PROMPT)] + state["messages"]
    )
    return {"messages": [response]}


langgraph_builder = StateGraph(MessagesState)
langgraph_builder.add_node("agent", call_model)
langgraph_builder.add_node("tools", ToolNode([retriever_tool], handle_tool_errors=True))

langgraph_builder.add_edge(START, "agent")
langgraph_builder.add_conditional_edges("agent", tools_condition)
langgraph_builder.add_edge("tools", "agent")

langgraph_agent = langgraph_builder.compile()
print("Compiled the explicit LangGraph agent loop.")

### Visualize the Explicit LangGraph Agent

This graph should look like the core agent loop: model node, tools node, and a conditional route between them.

In [ ]:
try:
    display(Image(langgraph_agent.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Could not render Mermaid PNG. Mermaid source follows:\n")
    print(langgraph_agent.get_graph().draw_mermaid())
    print(f"\nRender error: {exc}")

### Stream the Explicit LangGraph Agent

This helper is intentionally similar to `print_agent_stream`. The difference is that we are streaming from the graph we built ourselves.

In [ ]:
def print_langgraph_stream(question: str):
    """Run the explicit LangGraph agent and print each graph update."""
    inputs = {"messages": [{"role": "user", "content": question}]}

    for chunk in langgraph_agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue

            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)

### Compare the Same Questions

Run the same style of questions through the explicit LangGraph agent. The exact wording may differ, but the retrieval decision should follow the same pattern.

In [ ]:
print_langgraph_stream("What urinary signs suggest my cat needs urgent veterinary care?")

In [ ]:
print_langgraph_stream("Who won the 2022 FIFA World Cup?")

#### ❓ Question #4

What parts did `create_agent` hide that the explicit LangGraph version made visible? When would you choose middleware, and when would you change the graph itself?

##### Answer:

_`create_agent` hid the **entire graph construction**: binding the tools to the model, the `agent`/`call_model` node, the `ToolNode` that actually executes the tool, the conditional routing that decides model-vs-tools (`tools_condition`), the `tools → agent` loop-back edge, the `START`/`END` wiring, and the `MessagesState` plumbing that threads messages through each step. The explicit LangGraph version makes all of that visible and editable — you can see and change every node and edge, including the conditional edge and the loop._

_**Choose middleware** for cross-cutting behavior that wraps the existing loop without changing its shape: logging/observability, model- or tool-call limits, retries, message trimming/injection, and guardrails. It's reusable across agents and keeps the topology intact._

_**Change the graph** when you need different control flow or state: adding new nodes (a relevance grader, a query-rewriter, a deterministic scope router), branching with conditional edges, bypassing the loop entirely (like Activity #2's `out_of_scope` route), custom state fields, or human-in-the-loop checkpoints. Rule of thumb: **middleware adjusts behavior *around* the loop; editing the graph changes the *shape* of the loop itself.**_

## 🏗️ Activity #2: Add Deterministic Scope Routing

The base LangGraph sends every question to the model and relies on the model to reject unrelated requests. Add a small deterministic route before the agent so clearly unrelated questions can bypass the model-tools loop.

### Requirements

1. Add an `out_of_scope` node that returns a brief scope message.
2. Add a routing function that sends likely cat-health questions to `agent` and clearly unrelated questions to `out_of_scope`.
3. Build and compile a new graph with the route immediately after `START`.
4. Test at least one cat-health question, one unrelated question, and one ambiguous question.
5. Explain where the deterministic route helps and where it is brittle.

**Further Reading:**
- [LangGraph Conditional Edges](https://docs.langchain.com/oss/python/langgraph/graph-api#conditional-edges)

In [ ]:
# Activity #2 workspace
# Add a deterministic scope route BEFORE the agent so clearly unrelated questions
# bypass the model+tools loop entirely.

from langchain_core.messages import AIMessage

OUT_OF_SCOPE_MESSAGE = (
    "This assistant is scoped to the cat health guideline corpus. "
    "I can help with cat preventive care, nutrition, hydration, vaccines, parasites, "
    "dental health, urinary warning signs, senior cats, stress, behavior, and safe "
    "home monitoring. Please ask a cat health question, and contact a veterinarian "
    "for any medical decisions."
)

# Simple keyword gate. Intentionally crude so we can see where it is brittle.
CAT_HEALTH_KEYWORDS = {
    "cat", "cats", "kitten", "kitty", "feline", "vet", "veterinarian", "vaccine",
    "vaccination", "parasite", "flea", "tick", "worm", "deworm", "dental", "teeth",
    "urinary", "litter", "nutrition", "diet", "food", "feeding", "hydration",
    "senior", "kidney", "vomit", "diarrhea", "grooming", "behavior", "stress",
    "symptom", "symptoms", "preventive", "purr", "meow", "claw", "spay", "neuter",
    "microchip", "paw", "whisker",
}


def _last_user_text(state: MessagesState) -> str:
    """Pull the most recent human message content out of the state."""
    for message in reversed(state["messages"]):
        if getattr(message, "type", None) == "human":
            content = message.content
            return content if isinstance(content, str) else str(content)
    return ""


def route_scope(state: MessagesState) -> str:
    """Deterministic router: cat-health -> 'agent', clearly unrelated -> 'out_of_scope'."""
    text = _last_user_text(state).lower()
    tokens = set(text.replace("?", " ").replace(".", " ").replace(",", " ").split())
    return "agent" if tokens & CAT_HEALTH_KEYWORDS else "out_of_scope"


def out_of_scope(state: MessagesState):
    """Return a brief scope message without calling the model or any tool."""
    return {"messages": [AIMessage(content=OUT_OF_SCOPE_MESSAGE)]}


# Reuse call_model and the retriever ToolNode from Task 6; add the out_of_scope node
# and a deterministic route right after START.
routed_builder = StateGraph(MessagesState)
routed_builder.add_node("agent", call_model)
routed_builder.add_node("tools", ToolNode([retriever_tool], handle_tool_errors=True))
routed_builder.add_node("out_of_scope", out_of_scope)

routed_builder.add_conditional_edges(
    START,
    route_scope,
    {"agent": "agent", "out_of_scope": "out_of_scope"},
)
routed_builder.add_conditional_edges("agent", tools_condition)
routed_builder.add_edge("tools", "agent")
routed_builder.add_edge("out_of_scope", END)

routed_agent = routed_builder.compile()
print("Compiled the scope-routed LangGraph agent.")


def print_routed_stream(question: str):
    """Stream the routed agent; the first node tells us which branch was taken."""
    print("\n" + "#" * 90)
    print(f"QUESTION: {question}")
    inputs = {"messages": [{"role": "user", "content": question}]}

    for chunk in routed_agent.stream(inputs, stream_mode="updates"):
        for node_name, update in chunk.items():
            print(f"\n--- Update from node: {node_name} ---")
            messages = update.get("messages", [])
            if not messages:
                print(update)
                continue

            latest_message = messages[-1]
            if isinstance(latest_message, ToolMessage):
                print(f"Tool result preview:\n{latest_message.content[:1200]}")
            elif hasattr(latest_message, "pretty_print"):
                latest_message.pretty_print()
            else:
                print(latest_message)


# 1. Clear cat-health question -> routes to 'agent' (and should retrieve).
print_routed_stream("What preventive care should an adult cat get each year?")

# 2. Clearly unrelated question -> routes to 'out_of_scope', bypassing the loop.
print_routed_stream("Who won the 2022 FIFA World Cup?")

# 3. Ambiguous question with no explicit cat keyword -> routes to 'out_of_scope'
#    even though it could be about a cat (a false negative that shows the brittleness).
print_routed_stream("Is it normal for him to sleep most of the day and eat less?")

### 📝 Activity #2 Notes

- Which questions bypassed the model-tools loop?
- What happened with the ambiguous question?
- What are the cost, latency, and quality trade-offs of this route?

##### Answer:

_**Bypassed the loop:** The clearly unrelated question ("Who won the 2022 FIFA World Cup?") matched no cat-health keyword, so `route_scope` sent it straight to `out_of_scope`. The stream shows only an `out_of_scope` node — **no `agent` (model) call and no `tools` retrieval** — returning the fixed scope message. The clear cat-health question ("annual preventive care") matched keywords, routed to `agent`, and went through the normal model → tools loop._

_**Ambiguous question:** "Is it normal for him to sleep most of the day and eat less?" is plausibly about a cat, but it contains no keyword from the list (it uses the pronoun "him"), so the router **wrongly sent it to `out_of_scope`** — a false negative. This is exactly where the deterministic gate is brittle: it can't resolve pronouns/context, synonyms, typos, or other languages, and the symmetric failure also exists (a non-cat question containing a word like "diet" or "stress" would be a false positive routed into the agent)._

_**Trade-offs:**_
- _**Cost & latency:** Big win for clearly out-of-scope traffic — those requests skip the LLM (and embeddings) entirely, giving a near-instant, deterministic, zero-token refusal instead of a model call. Routing itself is essentially free._
- _**Quality:** The crude keyword match misclassifies in both directions, so it can reject real cat questions (hurting recall) or admit junk. The base graph's model-judged scoping is more robust to phrasing but costs a model call._
- _**Maintainability:** A hardcoded keyword set drifts and is hard to keep complete._

_Better production options keep the cheap-bypass benefit while reducing brittleness: a small/fast classifier model, or an embedding-similarity gate against the corpus, as the router instead of literal keywords._

## 🚧 Advanced Build (Optional): Add Explicit Retrieval Quality Control

The base assignment shows the minimal agentic RAG loop two ways:

1. `create_agent`
2. explicit LangGraph

If you want more control, extend the explicit LangGraph version with retrieval quality control. Good advanced additions include:

- Add a document relevance grader after retrieval.
- Add a query rewrite node when retrieval is weak.
- Add a loop limit so the agent cannot keep retrying forever.
- Add a deterministic guardrail before answering.

Those are useful production patterns, but they are not required for the core idea of agentic RAG.

---
## Summary

In this session, you:

1. Built a retrieval tool over a Qdrant vector store.
2. Used `create_agent` and middleware for a high-level agentic RAG loop.
3. Streamed agent runs to inspect when retrieval happened.
4. Rebuilt the loop explicitly with LangGraph nodes and conditional edges.
5. Practiced choosing between middleware-level constraints and graph-level routing.

### Key Takeaways

- Agentic RAG makes retrieval an available action instead of a mandatory pre-step.
- Tool contracts and system prompts strongly influence retrieval decisions.
- Middleware is useful for cross-cutting behavior such as logging, limits, retries, and guardrails.
- Explicit graphs are useful when the application needs custom state or control flow.
- Inspecting intermediate events is essential because a plausible final answer can hide a poor agent path.

### Further Reading

- [LangChain Agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangChain Middleware](https://docs.langchain.com/oss/python/langchain/middleware)
- [LangGraph Overview](https://docs.langchain.com/oss/python/langgraph/overview)
- [LangSmith Observability](https://docs.langchain.com/langsmith/observability)

### Notebook Output Guidance

Keep useful outputs when you submit, especially graph diagrams and representative streamed runs that support your observations. Remove secrets, failed experiments that no longer matter, and excessively noisy output.